<a href="https://colab.research.google.com/github/pelinbalci/SLM-FineTune/blob/main/Part_8_1_Unsloth_FineTune_Deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Part 8: Unsloth FineTune & GGUF

**You've fine-tuned a model. Now what?**

This notebook covers how to actually **use** your fine-tuned model.

---

## What We'll Cover

| Topic | Description |
|-------|-------------|
| Quick Training | Train a model with Unsloth (fast!) |
| Local Inference | Run the model directly |
| Saving Formats | LoRA, merged, GGUF |
| HuggingFace Hub | Upload and share |
| GGUF + llama.cpp | Run on CPU/edge devices |
| Ollama Integration | Easy local deployment |

---

## Deployment Options Overview

```
┌─────────────────────────────────────────────────────────────────┐
│                    YOUR FINE-TUNED MODEL                        │
└─────────────────────────┬───────────────────────────────────────┘
                          │
          ┌───────────────┼───────────────┐
          │               │               │
          ▼               ▼               ▼
   ┌─────────────┐ ┌─────────────┐ ┌─────────────┐
   │   LoRA      │ │   Merged    │ │    GGUF     │
   │  Adapter    │ │   Model     │ │  (llama.cpp)│
   └──────┬──────┘ └──────┬──────┘ └──────┬──────┘
          │               │               │
          ▼               ▼               ▼
   ┌─────────────┐ ┌─────────────┐ ┌─────────────┐
   │ HuggingFace │ │   vLLM /    │ │   Ollama    │
   │ Inference   │ │   TGI       │ │   Local     │
   └─────────────┘ └─────────────┘ └─────────────┘
```

---

# Part A: Setup & Quick Training

Let's train a model quickly with Unsloth so we have something to deploy. See more about thetraining details on: https://medium.com/@balci.pelin/unsloth-vs-standard-training-92d4c35b8ad8

In [ ]:
!pip install -q -U unsloth

In [ ]:
import unsloth
from unsloth import FastLanguageModel

In [ ]:
# Configuration - same for both experiments
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_SEQ_LENGTH = 512
BATCH_SIZE = 2
GRAD_ACCUM = 4
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4

# LoRA settings
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Model
model, tokenizer_unsloth = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized checkpointing
    random_state=42,
)

In [ ]:
# Data
from datasets import load_dataset
dataset = load_dataset("mlabonne/guanaco-llama2-1k", split="train")

from transformers import DataCollatorForSeq2Seq
from transformers import TrainingArguments

def tok_fn(examples):
    out = tokenizer_unsloth(
        examples["text"],
        truncation=True,   # ✅ Cut sequences > MAX_SEQ_LENGTH
        max_length=MAX_SEQ_LENGTH,
        padding=False,          # important: no padding in dataset
    )
    return out

tokenized_dataset = dataset.map(tok_fn, batched=True, remove_columns=dataset.column_names)

In [ ]:
# Data collator with proper truncation
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer_unsloth,
    padding=True,
    pad_to_multiple_of=8,
)

training_args = TrainingArguments(
    output_dir="./output",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    fp16=True,
    bf16=False,
    logging_steps=50,
    save_strategy="no",
    optim="adamw_8bit",
    warmup_ratio=0.03,
    report_to="none",
)

from trl import SFTTrainer

trainer_unsloth = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer_unsloth,
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    data_collator=data_collator,
)

result_unsloth = trainer_unsloth.train()

In [ ]:
result_unsloth

---

# Part B: Local Inference

The simplest way to use your model: run it directly!

In [ ]:
# Enable fast inference mode
FastLanguageModel.for_inference(model)
print("✅ Inference mode enabled")

In [ ]:
def chat(prompt, max_tokens=256):
    """Simple chat function."""
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer_unsloth.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_tokens,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer_unsloth.pad_token_id,
    )

    response = tokenizer_unsloth.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response

print("✅ chat() function ready")

In [ ]:
# Test the model
print("🧪 Testing the fine-tuned model:\n")

test_prompts = [
    "What is machine learning?",
    "Write a Python function to reverse a string.",
    "Give me 3 tips for better sleep.",
]

for prompt in test_prompts:
    print(f"👤 User: {prompt}")
    response = chat(prompt)
    print(f"🤖 Assistant: {response}")
    print("-" * 50)

---

# Part C: Saving Your Model

Multiple formats for different use cases:

| Format | Size | Use Case |
|--------|------|----------|
| LoRA adapter | ~10-50 MB | Share adapters, combine with base |
| Merged 16-bit | ~1-14 GB | HuggingFace, full precision |
| Merged 4-bit | ~0.5-4 GB | Smaller, quantized |
| GGUF | ~0.3-8 GB | llama.cpp, Ollama, CPU inference |

## Option 1: Save as GGUF (for llama.cpp / Ollama)

GGUF is the format used by:
- **llama.cpp** — C++ inference engine
- **Ollama** — Easy local LLM runner
- **LM Studio** — Desktop app for LLMs
- **GPT4All** — Local LLM platform

### Quantization Options

| Method | Bits | Size | Quality | Speed |
|--------|------|------|---------|-------|
| q8_0 | 8-bit | Large | Best | Slow |
| q5_k_m | 5-bit | Medium | Great | Medium |
| q4_k_m | 4-bit | Small | Good | Fast |
| q3_k_m | 3-bit | Tiny | OK | Fastest |
| q2_k | 2-bit | Smallest | Poor | Fastest |




Unsloth provides the adapter: save_pretrained_gguf.

In [ ]:
import os

# Save as GGUF with different quantizations
GGUF_PATH = "./my_model_gguf"

# q4_k_m is a good balance of size and quality
model.save_pretrained_gguf(
    GGUF_PATH,
    tokenizer_unsloth,
    quantization_method="q4_k_m",
)

print(f"\n✅ GGUF model saved!")

# List GGUF files
print(f"\n📁 GGUF files:")
for f in os.listdir(GGUF_PATH):
    if f.endswith('.gguf'):
        size = os.path.getsize(os.path.join(GGUF_PATH, f)) / 1e9
        print(f"   {f}: {size:.2f} GB")

---

# Part D: Upload to HuggingFace Hub

Share your model with the world!

In [ ]:
# Login to HuggingFace (you'll need a token)
# Get your token at: https://huggingface.co/settings/tokens

from huggingface_hub import login

# Option 1: Login interactively
login()

# Option 2: Use token directly (uncomment and add your token)
# login(token="hf_your_token_here")

print("💡 To upload, run: login() or login(token='your_token')")

In [ ]:
# Upload options (uncomment to use)

HUB_USERNAME = "pelinbalci"  # Change this!
MODEL_NAME = "my-qwen-finetuned"


# Upload GGUF (for Ollama users)
model.push_to_hub_gguf(
    f"{HUB_USERNAME}/{MODEL_NAME}-gguf",
    tokenizer_unsloth,
    quantization_method="q4_k_m",
)

---

# PART E - Use The Model

1- Ollama

Ollama can pull GGUF models directly from HuggingFace.

2- LM Studio




